____
### 1. Imports

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

____
### 2. Environment

Creating a custom environment to simulate the following condtions:
1. 30 day period
2. Required to sell inventory of 100 units
3. Unit cost of each unit set at $5
4. Agents in the environment are required to maximise profit in the 30 day period


Methods (1 - 3 are required for all environments):
1. `__init__()` → setup
2. `reset()` → start a new episode
3. `step(action)` → apply an action
4. `demand(price)` → calculates the demand given a price

Attributes:
1. observation space, 5 numbers as an array [Leftover Stock, Days Left, Last known demand]
2. action space (price of good, number over a continuouse range from 5 to 50)
3. max steps (days of simulation)
4. cost (cost incurred to obtain 1 unit)


In [2]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory, self.max_steps - self.step_count,
                         self.last_demand], dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 1.2
        noise = np.random.normal(0, 3) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
#### 2.1 Exploring the environment

In [3]:
env = DynamicPricingEnv()

In [4]:
obs, info = env.reset()
print(f"Observation space representing: [stock left, days left, last known sold]: {obs}")
print(f"Extra info dictrionary: {info}")

obs, reward, terminated, truncated, info = env.step(10)
env.get_latest()
print(reward)
print(f"{terminated}, {truncated}")

Observation space representing: [stock left, days left, last known sold]: [100.  30.   0.]
Extra info dictrionary: {}
Leftover Stock: 70.0 units, Days Left 29.0, Sold Units: 30.0
150.0
False, False


____
### 3. Deciding which model to use 
- A standard Q-table cannot be used as the action space (price of good) is a continuous number instead of a discrete number

#### 3.1 Proximal Policy Optimization (PPO) Model
- a policy gradient method which directly learns "Given this state, what price should I output"
- the "Proximal" part of the PPO model adjusts the actions in small amounts to find the optimal policy 
- therefore, PPO Models limits how drastically the policy changes each update to prevent unstable training

##### PPO Architecture
- PPO uses 2 networks
1. Actor Network
    - Outputs the pricing policy
    - Given a state, feeds into a neutal network and outputs a pricing distribution
    - `state → neural network → price distribution`
    - The agent then samples prices around that range

2. Critic Network
    - Estimates the future rewards
    - Asks "How profitable is this situation" and helps the Actor Network improve

##### PPO Flow
`Observe market → Choose price → Simulate customer response → Get profit reward → Update pricing policy slightly`


#### 3.2 Twin Delayed Deep Deterministic Policy Gradient (TD3)
- Designed specefically for continuous action space, precise control and stable deep Q-learning
- instead learning "what action should I take?" the model learns "how good is a particular action"

##### TD3 Architecture
- TD3 uses 3 networks
1. Actor Network
    - Outputs a distribution over actions
    - `state → distribution`
    - Samples from the distribution to create exploration

2. Critic Network (2 critic networks)
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as TD3)

3. Replay Buffer
    - To store experiences and reuse them, making the SAC highly sample efficient
    - Allow SAC to learn from past experiences repeatedly

4. Target Networks

##### TD3 Flow
- `Observe State → Actor suggests an action → Twin Critic Networks evaluate long term reward of the actions → Lower Q-value generated from the 2 networks is used`
- Therefore, the lower Q-value is used to train the critics and the actor (actor updated occassionally)



#### 3.3 Soft Actor-Critic (SAC)
- Considered one of the strongest RL algorithms for continuous control
- Combines actor-critic learning, entropy maximisation and off-policy training
- Tries to maximise both `Reward` and `Exploration` instead of only `profit`
- Entropy refers to the epsilon (randomness of the actions) therefore it encourages the agent to keep exploring pricing options, preventing the model from becoming too deterministic 


##### SAC Architecture
- SAC uses 3 networks
1. Actor Network
    - Outputs the exact price
    - `state → price`

2. Critic Network (2 critic networks)
    - The "Twin" part is in refernce to the 2 critic networks
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` 

##### SAC Flow
- `Observe State → Sample action from policy distribution → receive reward → update critic → update actor → encourage exploration through entropy bonus`
- Reward is evaluated as `total reward = reward + entropy bonus` and to encourage exploration

____
### 4. Training the model 
- In the spirit of learning, I will be training a PPO model, a TD3 model and a SAC model

#### 4.1 PPO Model
- Actor and Critic networks are first created

In [ ]:
class ActorCritic(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    """
    Shared backbone → splits into actor (policy) and critic (value).
    Actor outputs mean + log_std for a Gaussian distribution over price.
    Critic outputs a single scalar value estimate.
    """
    def __init__(self, obs_dim, action_dim):
        super().__init__() #initialises the parent nn.Module class
        self.backbone = nn.Sequential( # acts as a shared extractor used by both the actor and critic network
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )

        # actor head — outputs mean of price distribution
        self.actor_mean = nn.Linear(64, action_dim)

        # log_std as a learnable parameter (not input-dependent)
        self.log_std = nn.Parameter(torch.zeros(action_dim))

        # critic head — estimates expected total reward from this state
        self.critic = nn.Linear(64, 1)

    def forward(self, obs):
        features = self.backbone(obs)
        mean      = self.actor_mean(features)
        std       = self.log_std.exp().expand_as(mean)
        value     = self.critic(features)
        return mean, std, value

    def get_action(self, obs):
        """Sample an action and return it with its log probability."""
        mean, std, value = self.forward(obs)
        dist    = Normal(mean, std)
        action  = dist.sample()
        log_prob = dist.log_prob(action).sum(dim=-1)
        return action, log_prob, value.squeeze(-1)

    def evaluate(self, obs, action):
        """Re-evaluate stored actions during the update step."""
        mean, std, value = self.forward(obs)
        dist     = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy  = dist.entropy().sum(dim=-1)
        return log_prob, value.squeeze(-1), entropy

- C

In [ ]:
class RolloutBuffer:
    """Stores one batch of experience before each PPO update."""

    def __init__(self):
        self.clear()

    def clear(self):
        self.obs, self.actions, self.log_probs = [], [], []
        self.rewards, self.values, self.dones  = [], [], []

    def add(self, obs, action, log_prob, reward, value, done):
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95):
        """Compute GAE advantages and discounted returns."""
        advantages = []
        gae        = 0.0
        values     = self.values + [last_value]

        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)

        returns    = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns

    def to_tensors(self, advantages, returns, device):
        obs        = torch.tensor(np.array(self.obs),      dtype=torch.float32).to(device)
        actions    = torch.stack(self.actions).to(device)
        log_probs  = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages,               dtype=torch.float32).to(device)
        returns    = torch.tensor(returns,                  dtype=torch.float32).to(device)

        # normalize advantages for training stability
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns